In [ ]:
!apt-get update && apt-get install -y ffmpeg
!pip install uv

In [ ]:
# Thay bằng link repository của bạn
!git clone https://github.com/ngocbao220/duplex_chat.git
%cd duplex_chat

In [ ]:
!uv sync
!uv add wrapt

In [ ]:
from google.colab import userdata
import os

hf_token = userdata.get("HF_TOKEN") or os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Set HF_TOKEN in Colab secrets before running this notebook.")
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
!hf auth login --token $HUGGING_FACE_HUB_TOKEN

In [ ]:
from pathlib import Path

AUDIO_PATH = "/kaggle/input/datasets/ngocbaotrinhtuan/inputs/real.wav"
OUT_ROOT = Path("/kaggle/working/model_audio_tests")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_TESTS = [
    {
        "name": "pyannote31__dialoguesidon",
        "install": "uv sync --extra diarization-pyannote --extra separation-dialoguesidon",
        "diarization_backend": "pyannote",
        "diarization_model": "pyannote/speaker-diarization-3.1",
        "separation_backend": "dialoguesidon",
        "separation_model": "sarulab-speech/DialogueSidon",
    },
    {
        "name": "sortformer__dialoguesidon",
        "install": "uv sync --extra diarization-sortformer --extra separation-dialoguesidon",
        "diarization_backend": "sortformer",
        "diarization_model": "nvidia/diar_sortformer_4spk-v1",
        "separation_backend": "dialoguesidon",
        "separation_model": "sarulab-speech/DialogueSidon",
    },
    {
        "name": "diarizen__dialoguesidon",
        "install": "uv pip install -r requirements/diarization-diarizen.txt && uv sync --extra separation-dialoguesidon",
        "diarization_backend": "diarizen",
        "diarization_model": "BUT-FIT/diarizen-wavlm-large-s80-md",
        "separation_backend": "dialoguesidon",
        "separation_model": "sarulab-speech/DialogueSidon",
    },
    {
        "name": "pyannote31__sepformer",
        "install": "uv sync --extra diarization-pyannote --extra separation-sepformer",
        "diarization_backend": "pyannote",
        "diarization_model": "pyannote/speaker-diarization-3.1",
        "separation_backend": "sepformer",
        "separation_model": "speechbrain/sepformer-wsj02mix",
    },
    {
        "name": "pyannote31__mossformer2",
        "install": "uv pip install -r requirements/separation-mossformer2.txt && uv sync --extra diarization-pyannote",
        "diarization_backend": "pyannote",
        "diarization_model": "pyannote/speaker-diarization-3.1",
        "separation_backend": "mossformer2",
        "separation_model": "alibabasglab/MossFormer2_SS_16K",
    },
]

# Chạy tất cả nếu muốn: ENABLED_TEST_NAMES = [test["name"] for test in MODEL_TESTS]
ENABLED_TEST_NAMES = ["pyannote31__dialoguesidon"]

for test in MODEL_TESTS:
    if test["name"] not in ENABLED_TEST_NAMES:
        continue
    out_dir = OUT_ROOT / test["name"]
    out_dir.mkdir(parents=True, exist_ok=True)
    output_prefix = out_dir / "speaker"
    print(f"\n=== Running {test['name']} ===")
    print(f"Audio will be saved under: {out_dir}")
    install_cmd = test.get("install")
    if install_cmd:
        !{install_cmd}
    !MPLBACKEND=Agg uv run python test_single.py "$AUDIO_PATH"         --diarize-chunk 240         --separate-chunk 240         --diarization-backend "{test['diarization_backend']}"         --diarization-model "{test['diarization_model']}"         --separation-backend "{test['separation_backend']}"         --separation-model "{test['separation_model']}"         --output-prefix "$output_prefix"


In [ ]:
from pathlib import Path
import IPython.display as ipd

AUDIO_PATH = "/kaggle/input/datasets/ngocbaotrinhtuan/inputs/real.wav"
OUT_ROOT = Path("/kaggle/working/model_audio_tests")

print("Audio gốc:")
display(ipd.Audio(AUDIO_PATH))

for out_dir in sorted(OUT_ROOT.glob("*")):
    spk_a = out_dir / "speaker_A.wav"
    spk_b = out_dir / "speaker_B.wav"
    if not spk_a.exists() or not spk_b.exists():
        continue
    print(f"\nModel test: {out_dir.name}")
    print("Giọng Người A:")
    display(ipd.Audio(str(spk_a)))
    print("Giọng Người B:")
    display(ipd.Audio(str(spk_b)))
